In [14]:

import os
import glob
import numpy as np
import nibabel as nib
from tqdm import tqdm
import sklearn # Required for train_test_split, ensure it's installed if not already

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.transforms import (
    Compose, Orientationd, Spacingd, CropForegroundd,
    ScaleIntensityRanged, NormalizeIntensityd, ToTensord,
    RandSpatialCropd, RandFlipd, RandGaussianNoised, RandAdjustContrastd,
    RandBiasFieldd, EnsureChannelFirstd
)
from monai.networks.nets import UNet, SegResNet
from monai.data import decollate_batch
from monai.metrics import DiceMetric
from monai.losses import DiceLoss, DiceCELoss

# Conditional import for intensity_normalization
_HAS_INTENSITY_NORMALIZATION = False
try:
    # Attempt import directly from top-level or other common sub-modules if 'normalize' doesn't work
    # Or, specifically target a version if the API changed.
    # The common path is intensity_normalization.normalize
    from intensity_normalization.normalize import nyul_train_standard_scale, nyul_apply_standard_scale
    _HAS_INTENSITY_NORMALIZATION = True
except ImportError as e:
    print(f"Warning: 'intensity-normalization.normalize' module or its specific functions (nyul_train_standard_scale, nyul_apply_standard_scale) not importable: {e}")
    print("Nyul & Udupa normalization will be skipped. Please ensure 'intensity-normalization' is properly installed (e.g., version 1.7.0) and its API matches.")

# Conditional import for itk-elastix (still challenging to install via pip)
_HAS_ITK_ELASTIX = False
try:
    import itk # itk-elastix is often used with itk directly
    _HAS_ITK_ELASTIX = True
except ImportError:
    print("Warning: 'itk-elastix' not directly importable. Robust registration features will be limited.")


class ISLESDataset3D(Dataset):
    def __init__(self, raw_data_root_dir, raw_mask_root_dir):
        self.samples = []
        print(f"Scanning for 3D samples in raw data root: {raw_data_root_dir}")

        subject_dirs = sorted(glob.glob(os.path.join(raw_data_root_dir, 'sub-*')))

        for subject_dir in subject_dirs:
            subject_id = os.path.basename(subject_dir)
            ses_dwi_dir = os.path.join(subject_dir, "ses-0001", "dwi")
            ses_anat_dir = os.path.join(subject_dir, "ses-0001", "anat")
            mask_dir = os.path.join(raw_mask_root_dir, subject_id, "ses-0001")

            dwi_files = glob.glob(os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_dwi.nii.gz'))
            adc_files = glob.glob(os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_adc.nii.gz'))
            flair_files = glob.glob(os.path.join(ses_anat_dir, f'{subject_id}_ses-0001_FLAIR.nii.gz'))
            mask_files = glob.glob(os.path.join(mask_dir, f'{subject_id}_ses-0001_msk.nii.gz'))

            if dwi_files and adc_files and flair_files and mask_files:
                self.samples.append({
                    "image_dwi": dwi_files[0],
                    "image_adc": adc_files[0],
                    "image_flair": flair_files[0],
                    "label": mask_files[0],
                    "subject_id": subject_id
                })
            else:
                print(f"Skipping subject {subject_id} due to missing files:")
                if not dwi_files: print(f"  Missing DWI: {os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_dwi.nii.gz')}")
                if not adc_files: print(f"  Missing ADC: {os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_adc.nii.gz')}")
                if not flair_files: print(f"  Missing FLAIR: {os.path.join(ses_anat_dir, f'{subject_id}_ses-0001_FLAIR.nii.')}")
                if not mask_files: print(f"  Missing Mask: {os.path.join(mask_dir, f'{subject_id}_ses-0001_lesion-msk.nii')}")

        print(f"Total 3D samples found: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_paths = self.samples[idx]

        try:
            # Load images as SimpleITK objects, then convert to NumPy arrays.
            # MONAI transforms will handle further spatial/intensity standardization.
            dwi_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_dwi"], sitk.sitkFloat32)).astype(np.float32)
            adc_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_adc"], sitk.sitkFloat32)).astype(np.float32)
            flair_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_flair"], sitk.sitkFloat32)).astype(np.float32)
            mask_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["label"], sitk.sitkUInt8)).astype(np.float32)

            mask_data = (mask_data > 0.5).astype(np.float32)

            image_stacked = np.stack([dwi_data, adc_data, flair_data], axis=0)

            return {"image": image_stacked, "label": mask_data, "subject_id": sample_paths["subject_id"]}

        except Exception as e:
            print(f"Error loading or processing sample {sample_paths.get('subject_id', 'N/A')}: {e}")
            raise

# --- Data Loading and Preprocessing (Step 1: Define Data Paths and Structure) ---

# Adjusting data_root based on your "refer previous directory ../data" instruction.
# This assumes your notebook is in a directory like `/content/project/notebooks/`
# and your ISLES-2022 data is located at `/content/project/data/ISLES-2022/`.

base_data_dir = os.path.join("/content/drive/My Drive/", 'data/ISLES-2022')

data_root =base_data_dir
mask_root_for_raw = os.path.join(base_data_dir, 'derivatives')

# This `preprocessed_data_root` is where *you would save* the output of offline preprocessing.
# The `ISLESDataset3D` in this version is *not* reading from here; it's reading from `data_root` (raw data).
preprocessed_data_root = os.path.join("/content/drive/My Drive/ISLES-2022_preprocessed")

print(f"Base Project Directory: {base_project_dir}")
print(f"Resolved Raw Data Root: {data_root}")
print(f"Resolved Raw Mask Root: {mask_root_for_raw}")
print(f"Conceptual Preprocessed Data Root: {preprocessed_data_root}")

os.makedirs(preprocessed_data_root, exist_ok=True)

full_dataset = ISLESDataset3D(data_root, mask_root_for_raw)

# Split indices for training and validation
from sklearn.model_selection import train_test_split
train_indices, val_indices = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=42)

# Create subset datasets
train_ds = torch.utils.data.Subset(full_dataset, train_indices)
val_ds = torch.utils.data.Subset(full_dataset, val_indices)

print(f"Total training subjects: {len(train_ds)}")
print(f"Total validation subjects: {len(val_ds)}")

# --- MONAI Transforms for Data Augmentation and Spatial Standardization ---
# These transforms will be applied to the NumPy arrays returned by ISLESDataset3D.__getitem__
# They handle channel addition, orientation, spacing, and intensity transforms.

target_spacing = (1.0, 1.0, 1.0) # Example target spacing for final data
roi_size = (128, 128, 128) # Example patch size for training

train_transforms = Compose(
    [
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=roi_size),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys="image", subtrahend=0.5, divisor=0.5),

        RandSpatialCropd(keys=["image", "label"], roi_size=roi_size, random_size=False, random_center=True),
        RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.5),
        RandGaussianNoised(keys=["image"], prob=0.1, std=0.01),
        RandAdjustContrastd(keys=["image"], prob=0.1, gamma=(0.7, 1.3)),
        RandBiasFieldd(keys=["image"], prob=0.1, coeff_range=(0.0, 0.05)),

        ToTensord(keys=["image", "label"]),
    ]
)

val_transforms = Compose(
    [
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=roi_size),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], subtrahend=0.5, divisor=0.5),
        ToTensord(keys=["image", "label"]),
    ]
)

# Apply MONAI transforms to the subset datasets
train_ds.transform = train_transforms
val_ds.transform = val_transforms

# Create MONAI DataLoaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

Nyul & Udupa normalization will be skipped. Please ensure 'intensity-normalization' is properly installed (e.g., version 1.7.0) and its API matches.
Base Project Directory: /
Resolved Raw Data Root: /content/drive/My Drive/data/ISLES-2022
Resolved Raw Mask Root: /content/drive/My Drive/data/ISLES-2022/derivatives
Conceptual Preprocessed Data Root: /content/drive/My Drive/ISLES-2022_preprocessed
Scanning for 3D samples in raw data root: /content/drive/My Drive/data/ISLES-2022
Total 3D samples found: 250
Total training subjects: 200
Total validation subjects: 50


In [ ]:
import os
import shutil
import numpy as np
import SimpleITK as sitk
from tqdm import tqdm
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def apply_n4_bias_correction(input_path, output_path):
    input_image = sitk.ReadImage(input_path)
    # Convert to float32
    input_image = sitk.Cast(input_image, sitk.sitkFloat32)

    # Try to get mask path
    mask_path = input_path.replace('.nii.gz', '_msk.nii.gz')
    if os.path.exists(mask_path):
        mask_image = sitk.ReadImage(mask_path)
    else:
        mask_image = None

    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    number_fitting_levels = 4

    # If no mask is available, just use the input image
    output_image = corrector.Execute(input_image, mask_image) if mask_image else corrector.Execute(input_image)

    # Convert back to original pixel type if needed
    output_image = sitk.Cast(output_image, sitk.sitkInt16)

    sitk.WriteImage(output_image, output_path)

def register_image_and_mask(fixed_image_path, moving_image_path, mask_path, output_dir, subject_id, modality):
    # Read images
    fixed_image = sitk.ReadImage(fixed_image_path)
    moving_image = sitk.ReadImage(moving_image_path)

    # Get original spacing
    fixed_spacing = fixed_image.GetSpacing()
    moving_spacing = moving_image.GetSpacing()

    # Fix spacing issues
    def fix_spacing(spacing):
        spacing = tuple(1.0 if x == 0 else x for x in spacing)
        spacing = tuple(abs(x) for x in spacing)
        spacing = tuple(max(x, 0.1) for x in spacing)
        return spacing

    fixed_spacing = fix_spacing(fixed_spacing)
    moving_spacing = fix_spacing(moving_spacing)

    # Set corrected spacing
    fixed_image.SetSpacing(fixed_spacing)
    moving_image.SetSpacing(moving_spacing)

    # Ensure images have the same dimension
    if fixed_image.GetDimension() != moving_image.GetDimension():
        raise ValueError(f"Fixed and moving images must have the same dimension. Fixed: {fixed_image.GetDimension()}, Moving: {moving_image.GetDimension()}")

    # Set up registration
    registration_method = sitk.ImageRegistrationMethod()

    # Set the metric
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)

    # Set the interpolator
    registration_method.SetInterpolator(sitk.sitkLinear)

    # Set the optimizer
    registration_method.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=100,
        estimateLearningRate=sitk.ImageRegistrationMethod.EachIteration
    )

    # Set initial transform
    initial_transform = sitk.CenteredTransformInitializer(
        fixed_image, moving_image,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    registration_method.SetInitialTransform(initial_transform)

    # Add logging
    registration_method.AddCommand(sitk.sitkStartEvent, lambda: logger.info(f"Start Registration {modality}"))
    registration_method.AddCommand(sitk.sitkIterationEvent, lambda: logger.info(f"Iteration {registration_method.GetOptimizerIteration()}"))

    # Cast to float32
    fixed_image = sitk.Cast(fixed_image, sitk.sitkFloat32)
    moving_image = sitk.Cast(moving_image, sitk.sitkFloat32)

    try:
        final_transform = registration_method.Execute(fixed_image, moving_image)

        # Log registration results
        logger.info(f"Final metric value: {registration_method.GetMetricValue()}")
        logger.info(f"Optimizer's stopping condition, {registration_method.GetOptimizerStopConditionDescription()}")

        # Get fixed image properties
        fixed_size = fixed_image.GetSize()
        fixed_spacing = fixed_image.GetSpacing()
        fixed_origin = fixed_image.GetOrigin()
        fixed_direction = fixed_image.GetDirection()

        # Create resampling filter
        resample = sitk.ResampleImageFilter()
        resample.SetSize(fixed_size)
        resample.SetOutputSpacing(fixed_spacing)
        resample.SetOutputOrigin(fixed_origin)
        resample.SetOutputDirection(fixed_direction)
        resample.SetTransform(final_transform)
        resample.SetInterpolator(sitk.sitkLinear)

        # Resample moving image
        resampled_image = resample.Execute(moving_image)

        # Save the registered image
        output_image_path = os.path.join(output_dir, f'{subject_id}_{modality}_registered.nii.gz')
        sitk.WriteImage(resampled_image, output_image_path)

        # If mask exists, apply the same transform to it
        if os.path.exists(mask_path):
            mask_image = sitk.ReadImage(mask_path)

            # Create mask resampling filter
            mask_resample = sitk.ResampleImageFilter()
            mask_resample.SetSize(fixed_size)
            mask_resample.SetOutputSpacing(fixed_spacing)
            mask_resample.SetOutputOrigin(fixed_origin)
            mask_resample.SetOutputDirection(fixed_direction)
            mask_resample.SetTransform(final_transform)
            mask_resample.SetInterpolator(sitk.sitkNearestNeighbor)

            resampled_mask = mask_resample.Execute(mask_image)
            sitk.WriteImage(resampled_mask, os.path.join(output_dir, f'{subject_id}_{modality}_mask_registered.nii.gz'))

        return output_image_path

    except Exception as e:
        logger.error(f"Registration failed for {modality}: {str(e)}")
        logger.error(f"Fixed image size: {fixed_image.GetSize()}")
        logger.error(f"Moving image size: {moving_image.GetSize()}")
        logger.error(f"Fixed image spacing: {fixed_image.GetSpacing()}")
        logger.error(f"Moving image spacing: {moving_image.GetSpacing()}")
        raise

def create_comprehensive_overlay(fixed_image_path, moving_image_path, registered_moving_path, mask_path, output_dir, subject_id):
    """
    Create a comprehensive overlay visualization.
    """
    try:
        logger.info(f"Creating overlay for {subject_id}")

        # Read images
        fixed_image = sitk.ReadImage(fixed_image_path)
        moving_image = sitk.ReadImage(moving_image_path)
        registered_moving = sitk.ReadImage(registered_moving_path)
        mask_image = sitk.ReadImage(mask_path)

        # Convert to numpy arrays
        fixed_array = sitk.GetArrayFromImage(fixed_image)
        moving_array = sitk.GetArrayFromImage(moving_image)
        registered_array = sitk.GetArrayFromImage(registered_moving)
        mask_array = sitk.GetArrayFromImage(mask_image)

        # Resample mask to match fixed image dimensions
        mask_resampled = sitk.Resample(mask_image, fixed_image,
                                      sitk.Transform(),
                                      sitk.sitkNearestNeighbor,
                                      0.0, mask_image.GetPixelID())
        mask_resampled_array = sitk.GetArrayFromImage(mask_resampled)

        # Create RGB visualization
        # Normalize images to 0-255 range
        fixed_array_norm = ((fixed_array - fixed_array.min()) /
                           (fixed_array.max() - fixed_array.min()) * 255).astype(np.uint8)
        registered_array_norm = ((registered_array - registered_array.min()) /
                               (registered_array.max() - registered_array.min()) * 255).astype(np.uint8)

        # Create RGB image
        # FLAIR in red channel
        # DWI in green channel
        # Mask in blue channel
        rgb_image = np.zeros((fixed_array.shape[0], fixed_array.shape[1], fixed_array.shape[2], 3), dtype=np.uint8)

        # Add FLAIR (red)
        rgb_image[..., 0] = fixed_array_norm

        # Add DWI (green)
        rgb_image[..., 1] = registered_array_norm

        # Add mask (blue)
        rgb_image[..., 2][mask_resampled_array > 0] = 255

        # Save RGB visualization
        rgb_image_sitk = sitk.GetImageFromArray(rgb_image)
        rgb_image_sitk.CopyInformation(fixed_image)
        sitk.WriteImage(rgb_image_sitk, os.path.join(output_dir, f'{subject_id}_flair_dwi_overlay.nii.gz'))

        logger.info(f"RGB overlay created for {subject_id}")

        # Create separate visualization files for each component
        # Save normalized FLAIR
        fixed_image_norm = sitk.GetImageFromArray(fixed_array_norm)
        fixed_image_norm.CopyInformation(fixed_image)
        sitk.WriteImage(fixed_image_norm, os.path.join(output_dir, f'{subject_id}_flair_normalized.nii.gz'))

        # Save normalized DWI
        registered_image_norm = sitk.GetImageFromArray(registered_array_norm)
        registered_image_norm.CopyInformation(fixed_image)
        sitk.WriteImage(registered_image_norm, os.path.join(output_dir, f'{subject_id}_dwi_normalized.nii.gz'))

        # Save mask overlay
        mask_overlay = np.zeros_like(fixed_array, dtype=np.uint8)
        mask_overlay[mask_resampled_array > 0] = 255
        mask_overlay_sitk = sitk.GetImageFromArray(mask_overlay)
        mask_overlay_sitk.CopyInformation(fixed_image)
        sitk.WriteImage(mask_overlay_sitk, os.path.join(output_dir, f'{subject_id}_mask_overlay.nii.gz'))

        logger.info(f"Individual overlays created for {subject_id}")

    except Exception as e:
        logger.error(f"Error creating overlay for {subject_id}: {str(e)}")

def preprocess_subject(subject_id, raw_data_root, mask_root_for_raw, preprocessed_data_root):
    try:
        logger.info(f"Processing {subject_id}")

        # Create output directory using absolute path
        output_subject_dir = os.path.abspath(os.path.join(preprocessed_data_root, subject_id, 'ses-0001'))
        os.makedirs(output_subject_dir, exist_ok=True)

        # Get input paths using absolute paths
        current_raw_dwi_path = os.path.abspath(os.path.join(raw_data_root, subject_id, 'ses-0001', 'dwi', f'{subject_id}_ses-0001_dwi.nii.gz'))
        current_raw_adc_path = os.path.abspath(os.path.join(raw_data_root, subject_id, 'ses-0001', 'dwi', f'{subject_id}_ses-0001_adc.nii.gz'))
        current_raw_flair_path = os.path.abspath(os.path.join(raw_data_root, subject_id, 'ses-0001', 'anat', f'{subject_id}_ses-0001_FLAIR.nii.gz'))

        # Use the correct mask file naming convention
        current_raw_mask_path = os.path.abspath(os.path.join(mask_root_for_raw, subject_id, 'ses-0001', f'{subject_id}_ses-0001_msk.nii.gz'))

        # Log paths for debugging
        logger.info(f"Output directory: {output_subject_dir}")
        logger.info(f"DWI path: {current_raw_dwi_path}")
        logger.info(f"ADC path: {current_raw_adc_path}")
        logger.info(f"FLAIR path: {current_raw_flair_path}")
        logger.info(f"Mask path: {current_raw_mask_path}")

        # Check if files exist
        if not all(os.path.exists(f) for f in [current_raw_dwi_path, current_raw_adc_path, current_raw_flair_path, current_raw_mask_path]):
            logger.warning(f"Skipping {subject_id}: Missing required files")
            logger.warning(f"  DWI: {os.path.exists(current_raw_dwi_path)}")
            logger.warning(f"  ADC: {os.path.exists(current_raw_adc_path)}")
            logger.warning(f"  FLAIR: {os.path.exists(current_raw_flair_path)}")
            logger.warning(f"  Mask: {os.path.exists(current_raw_mask_path)}")
            logger.warning(f"  Mask path: {current_raw_mask_path}")
            return

        # Apply N4 bias correction
        logger.info(f"Processing {subject_id}: N4 Bias Correction")

        # Create output paths using absolute paths
        n4_dwi_path = os.path.abspath(os.path.join(output_subject_dir, f'{subject_id}_dwi_n4.nii.gz'))
        n4_adc_path = os.path.abspath(os.path.join(output_subject_dir, f'{subject_id}_adc_n4.nii.gz'))
        n4_flair_path = os.path.abspath(os.path.join(output_subject_dir, f'{subject_id}_flair_n4.nii.gz'))

        # Log output paths for debugging
        logger.info(f"N4 DWI output: {n4_dwi_path}")
        logger.info(f"N4 ADC output: {n4_adc_path}")
        logger.info(f"N4 FLAIR output: {n4_flair_path}")

        apply_n4_bias_correction(current_raw_dwi_path, n4_dwi_path)
        apply_n4_bias_correction(current_raw_adc_path, n4_adc_path)
        apply_n4_bias_correction(current_raw_flair_path, n4_flair_path)

        # Copy mask file
        shutil.copy(current_raw_mask_path, os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'))

        # Registration
        logger.info(f"Processing {subject_id}: Registration")
        fixed_image_for_reg = n4_flair_path

        # Register FLAIR itself
        register_image_and_mask(
            fixed_image_for_reg,
            n4_flair_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'flair'
        )

        # Register DWI
        register_image_and_mask(
            fixed_image_for_reg,
            n4_dwi_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'dwi'
        )
        # Register DWI
        register_image_and_mask(
            fixed_image_for_reg,
            n4_dwi_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'dwi'
        )

        # Register ADC
        register_image_and_mask(
            fixed_image_for_reg,
            n4_adc_path,
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id,
            'adc'
        )

        # Generate visualizations
        logger.info(f"Processing {subject_id}: Generating visualizations")
        create_comprehensive_overlay(
            n4_flair_path,
            n4_dwi_path,
            os.path.join(output_subject_dir, f'{subject_id}_dwi_registered.nii.gz'),
            os.path.join(output_subject_dir, f'{subject_id}_lesion-msk.nii.gz'),
            output_subject_dir,
            subject_id
        )

    except Exception as e:
        logger.error(f"Error processing subject {subject_id}: {str(e)}")
        raise
        logger.error(f"Error processing subject {subject_id}: {str(e)}")
        raise

def main_preprocessing():
    # Define absolute paths
    base_data_dir = os.path.abspath("../data")
    raw_data_root = base_data_dir
    mask_root_for_raw = os.path.join(base_data_dir, 'derivatives')
    preprocessed_data_root = os.path.abspath("../data/ISLES-2022_preprocessed")

    # Create preprocessed root directory
    os.makedirs(preprocessed_data_root, exist_ok=True)

    # Get list of subjects
    subjects = [d for d in os.listdir(raw_data_root) if d.startswith('sub-')]

    # Process each subject
    for subject_id in tqdm(subjects, desc="Preprocessing"):
        preprocess_subject(subject_id, raw_data_root, mask_root_for_raw, preprocessed_data_root)

if __name__ == "__main__":
    main_preprocessing()

Preprocessing:   0%|          | 0/199 [00:00<?, ?it/s]INFO:__main__:Processing sub-strokecase0051
INFO:__main__:Output directory: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\ISLES-2022_preprocessed\sub-strokecase0051\ses-0001
INFO:__main__:DWI path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0051\ses-0001\dwi\sub-strokecase0051_ses-0001_dwi.nii.gz
INFO:__main__:ADC path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0051\ses-0001\dwi\sub-strokecase0051_ses-0001_adc.nii.gz
INFO:__main__:FLAIR path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\sub-strokecase0051\ses-0001\anat\sub-strokecase0051_ses-0001_FLAIR.nii.gz
INFO:__main__:Mask path: c:\Users\punit\OneDrive\MS_research_docs\FINAL_SUBMISSION\isles_data_set\data\derivatives\sub-strokecase0051\ses-0001\sub-strokecase0051_ses-0001_msk.nii.gz
INFO:__main__:Processing sub-strok

TypeError: in method 'ResampleImageFilter_SetOutputOrigin', argument 2 of type 'std::vector< double,std::allocator< double > >'

In [9]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm, Act

class MultiEncoderUNet(nn.Module):
    def __init__(self, in_channels_dwi, in_channels_adc, in_channels_flair, out_channels,
                 spatial_dims=3, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2)):
        super().__init__()
        # Separate encoders for each modality [2, 32, 33, 46]
        self.encoder_dwi = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_dwi,
            out_channels=out_channels, # This output is just before bottleneck
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder # Access the encoder part

        self.encoder_adc = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_adc,
            out_channels=out_channels,
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder

        self.encoder_flair = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_flair,
            out_channels=out_channels,
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder

        # Shared decoder (output channels for each encoder are summed at bottleneck)
        # The input channels to the decoder will be sum of bottleneck channels from all encoders
        # For simplicity, let's assume the last channel in 'channels' is the bottleneck feature size
        bottleneck_features = channels[-1] * 3 # Assuming 3 modalities
        self.decoder = UNet(
            spatial_dims=spatial_dims,
            in_channels=bottleneck_features,
            out_channels=out_channels,
            channels=channels[::-1], # Reverse channels for decoder
            strides=strides[::-1], # Reverse strides for decoder
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU,
            is_decoder=True # Indicate this is the decoder part
        ) # This is a simplified representation. A true MultiEncoderUNet would manage skip connections carefully.

    def forward(self, x):
        # x is expected to be a dictionary or a concatenated tensor
        # For this simplified example, assume x is already concatenated
        # In a real MONAI pipeline, you'd pass a dictionary and handle it with transforms
        # For demonstration, let's assume input x has 3 channels for DWI, ADC, FLAIR
        dwi_input = x[:, 0:1, :, :, :] # Assuming channel 0 is DWI
        adc_input = x[:, 1:2, :, :, :] # Assuming channel 1 is ADC
        flair_input = x[:, 2:3, :, :, :] # Assuming channel 2 is FLAIR

        # Encode each modality
        dwi_features = self.encoder_dwi(dwi_input)
        adc_features = self.encoder_adc(adc_input)
        flair_features = self.encoder_flair(flair_input)

        # Concatenate features at bottleneck (simplified, actual nnU-Net handles skip connections)
        # This is a conceptual bottleneck fusion. Actual Multi-encoder nnU-Net combines features
        # from corresponding encoder layers to feed into the decoder's skip connections.
        # For this simplified UNet decoder, we'll just concatenate the deepest features.
        fused_bottleneck = torch.cat([dwi_features, adc_features, flair_features], dim=1) #  for deepest features

        # Pass through shared decoder
        # This is a placeholder. A true UNet decoder requires skip connections from encoders.
        # For a full implementation, consider adapting MONAI's UNet or SegResNet to accept
        # multiple encoder outputs and merge them into the decoder's skip pathways.
        # For now, we'll pass the fused bottleneck into a simple decoder.
        # A more accurate implementation would require custom UNet structure or MONAI's support for multi-input.
        # For this example, let's use a standard UNet and assume the input has 3 channels (DWI, ADC, FLAIR)
        # and the UNet handles the multi-channel input directly as a single input.
        # The "Multi-encoder" part would be handled by the data pipeline preparing the input.
        # The previous section's `ConcatItemsd` already creates a 3-channel input.
        # So, we revert to a standard UNet for simplicity of demonstration,
        # but note that Multi-encoder nnU-Net is conceptually superior.
        pass

# Revert to standard MONAI UNet for demonstration, assuming concatenated input
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = UNet(
    spatial_dims=3,
    in_channels=3, # 3 modalities: DWI, ADC, FLAIR
    out_channels=1, # Binary segmentation: lesion/non-lesion
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
    act=Act.LEAKYRELU
).to(device)

# For a true Multi-encoder nnU-Net, one would build a custom network that takes
# a dictionary of images (e.g., {'dwi': tensor, 'adc': tensor, 'flair': tensor})
# and processes them through separate encoders before combining features.
# This is more complex than a simple UNet and would require a custom MONAI network definition.

In [10]:
loss_function = DiceCELoss(to_onehot_y=False, sigmoid=True) # Combines Dice Loss and Cross-Entropy Loss [35, 36, 44]
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
dice_metric = DiceMetric(include_background=False, reduction="mean") # For evaluation [40]

In [18]:
# Import MONAI and other required packages
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ResizeWithPadOrCropd,
    ScaleIntensityRanged,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    RandGaussianNoised,
    RandScaleIntensityd,
    RandShiftIntensityd,
    ConcatItemsd,
    ToTensord
)
from monai.data import (
    Dataset,
    CacheDataset,
    DataLoader,
    list_data_collate
)
from sklearn.model_selection import train_test_split
import torch
import numpy as np
from pathlib import Path
import random

# Define data loading function
def load_preprocessed_data(preprocessed_root):
    data_list = []
    for subject_dir in Path(preprocessed_root).glob("sub-*"):
        subject_id = subject_dir.name
        data_list.append({
            "dwi": str(subject_dir / "ses-0001" / f"{subject_id}_dwi_n4.nii.gz"),
            "adc": str(subject_dir / "ses-0001" / f"{subject_id}_adc_n4.nii.gz"),
            "flair": str(subject_dir / "ses-0001" / f"{subject_id}_flair_n4.nii.gz"),
            "label": str(subject_dir / "ses-0001" / f"{subject_id}_lesion-msk.nii.gz"),
            "subject_id": subject_id
        })
    return data_list

# Load preprocessed data
preprocessed_data = load_preprocessed_data(preprocessed_data_root)
print(f"preprocessed_data_root",preprocessed_data_root)

# Split data into train and validation sets
train_data, val_data = train_test_split(preprocessed_data, test_size=0.2, random_state=42)

# Enhanced training transforms
train_transforms = Compose([
    LoadImaged(keys=["dwi", "adc", "flair", "label"]),
    EnsureChannelFirstd(keys=["dwi", "adc", "flair", "label"]),
    Orientationd(keys=["dwi", "adc", "flair", "label"], axcodes="RAS"),
    Spacingd(
        keys=["dwi", "adc", "flair", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=["bilinear", "bilinear", "bilinear", "nearest"],
        padding_mode=["zeros", "zeros", "zeros", "zeros"],
        align_corners=[True, True, True, True]
    ),
    ResizeWithPadOrCropd(
        keys=["dwi", "adc", "flair", "label"],
        spatial_size=(224, 224, 224),
        mode=["constant", "constant", "constant", "constant"]
    ),
    ScaleIntensityRanged(keys=["dwi", "adc", "flair"], a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["dwi", "adc", "flair"], subtrahend=0.5, divisor=0.5),

    # Advanced augmentations
    RandCropByPosNegLabeld(
        keys=["dwi", "adc", "flair", "label"],
        label_key="label",
        spatial_size=(128, 128, 128),
        pos=3,
        neg=1,
        num_samples=4
    ),

    # Basic spatial augmentations
    RandFlipd(keys=["dwi", "adc", "flair", "label"], spatial_axis=0, prob=0.5),
    RandFlipd(keys=["dwi", "adc", "flair", "label"], spatial_axis=1, prob=0.5),
    RandFlipd(keys=["dwi", "adc", "flair", "label"], spatial_axis=2, prob=0.5),
    RandRotate90d(keys=["dwi", "adc", "flair", "label"], prob=0.5, spatial_axes=(0, 1)),
    RandRotate90d(keys=["dwi", "adc", "flair", "label"], prob=0.5, spatial_axes=(1, 2)),
    RandRotate90d(keys=["dwi", "adc", "flair", "label"], prob=0.5, spatial_axes=(0, 2)),

    # Intensity augmentations
    RandGaussianNoised(keys=["dwi", "adc", "flair"], prob=0.1, mean=0.0, std=0.01),
    RandScaleIntensityd(keys=["dwi", "adc", "flair"], factors=0.1, prob=0.1),
    RandShiftIntensityd(keys=["dwi", "adc", "flair"], offsets=0.1, prob=0.1),

    ConcatItemsd(keys=["dwi", "adc", "flair"], name="image"),
    ToTensord(keys=["image", "label"])
])

# Create datasets
train_dataset = CacheDataset(
    data=train_data,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=4
)

val_dataset = CacheDataset(
    data=val_data,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=4
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate
)

# Define the model
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
import torch.optim as optim

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = UNet(
    spatial_dims=3,
    in_channels=3,  # DWI, ADC, FLAIR
    out_channels=2,  # Background and lesion
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2
).to(device)

# Loss function
loss_function = DiceCELoss(to_onehot_y=True, sigmoid=True, squared_pred=True)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Dice metric
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Training parameters
num_epochs = 100
val_interval = 5
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

# Training loop
# Post-processing transforms
post_pred = Compose([
    Activations(sigmoid=True),
    AsDiscrete(threshold_values=True)
])

# For labels, we need to ensure they're in the correct format
post_label = Compose([
    AsDiscrete(to_onehot=2)  # Convert to one-hot with 2 classes
])

# Update the validation loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"Epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)

                # Sliding window inference
                roi_size = (128, 128, 128)
                sw_batch_size = 4
                val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)

                # Post-process predictions and labels
                val_outputs = post_pred(val_outputs)
                val_labels = post_label(val_labels)

                # Update dice metric
                dice_metric(y_pred=val_outputs, y=val_labels)

            # Calculate and print metrics
            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)

            # Update learning rate
            scheduler.step(metric)

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_metric_model.pth")
                print("saved new best metric model")

            print(f"Current epoch: {epoch + 1} current mean dice: {metric:.4f}")
            print(f"Best mean dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

preprocessed_data_root /content/drive/My Drive/ISLES-2022_preprocessed


Loading dataset:  16%|█▌        | 7/44 [00:09<00:51,  1.38s/it]


RuntimeError: applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x7d59c32dba50>

In [ ]:
# Add these imports at the top with other imports
from monai.transforms import AsDiscrete
from monai.data import decollate_batch
import matplotlib.pyplot as plt
import os

metric = dice_metric.aggregate().item()
dice_metric.reset()
metric_values.append(metric)

if metric > best_metric:
    best_metric = metric
    best_metric_epoch = epoch + 1
    torch.save(model.state_dict(), "best_metric_model.pth")
     print("saved new best metric model")

            print(f"Current epoch: {epoch + 1} current mean dice: {metric:.4f}")
            print(f"Best mean dice: {best_metric:.4f} at epoch: {best_metric_epoch}")

# Visualization function
def visualize_results(image, label, pred, subject_id, slice_idx=112):
    plt.figure("check", (18, 6))

    plt.subplot(1, 3, 1)
    plt.title("image (DWI)")
    plt.imshow(image[0, 0, :, :, slice_idx], cmap="gray")

    plt.subplot(1, 3, 2)
    plt.title("label")
    plt.imshow(label[0, 0, :, :, slice_idx])

    plt.subplot(1, 3, 3)
    plt.title("prediction")
    plt.imshow(pred[0, 0, :, :, slice_idx])

    plt.savefig(f"results/{subject_id}_results.png")
    plt.close()

# Post-processing function
def postprocess_prediction(prediction):
    # Apply thresholding
    prediction = (prediction > 0.5).float()

    # Remove small connected components
    prediction = prediction.cpu().numpy()
    prediction = remove_small_objects(prediction, min_size=100)

    # Fill holes
    prediction = binary_fill_holes(prediction)

    return torch.from_numpy(prediction).float().to(prediction.device)

# Add this after the training loop
# Create results directory
os.makedirs("results", exist_ok=True)

# Test the model with the best weights
model.load_state_dict(torch.load("best_metric_model.pth"))
model.eval()

with torch.no_grad():
    for val_data in val_loader:
        val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
        roi_size = (128, 128, 128)
        sw_batch_size = 4

        # Sliding window inference
        val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model)
        val_outputs = torch.sigmoid(val_outputs)

        # Post-process predictions
        val_outputs = postprocess_prediction(val_outputs)

        # Calculate dice score
        dice_score = dice_metric(y_pred=val_outputs, y=val_labels)
        print(f"Subject {val_data['subject_id'][0]} Dice score: {dice_score.item():.4f}")

        # Save visualization
        visualize_results(
            val_inputs.cpu().numpy(),
            val_labels.cpu().numpy(),
            val_outputs.cpu().numpy(),
            val_data["subject_id"][0]
        )

IndentationError: unexpected indent (4008419763.py, line 8)